# Session 2: equilibrium of every pin gives every bar force
**A. Real-world problem.** A 6 m footbridge truss carries a 10 kN load at midspan. Before
anyone cuts steel, an engineer must know the force in every bar: which are pulled, which
are pushed, and how big. Here we do that with statics only.

**Plan.** (1) count bars, joints and supports to see whether statics is enough,
(2) write equilibrium of each pin (sum Fx = 0, sum Fy = 0) as one linear system,
(3) solve a three-bar bracket and a Warren truss, (4) check with the method of sections,
(5) size the bars for a stress limit.

Educational model: ideal pins, straight weightless bars, loads only at joints. Never use
it for a real structure.

In [ ]:
import pathlib

import numpy as np

# where figures are saved (works as a script and in Jupyter/Colab)
OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

## Step 1: determinacy, m + r = 2 j
m bars, r support reactions, j joints. Each joint gives 2 equations (sum Fx, sum Fy), so
there are 2 j equations for m + r unknowns. If m + r = 2 j, statics alone is enough
(statically determinate). If m + r < 2 j the truss is a mechanism (it can move). If
m + r > 2 j there are redundant bars (statically indeterminate): statics is not enough
and Session 3 handles that with stiffness.

In [ ]:
def determinacy(m, r, j):
    """Return 'determinate', 'mechanism' or 'indeterminate' from m + r compared with 2 j."""
    if m + r == 2 * j:
        return "determinate"
    return "mechanism" if m + r < 2 * j else "indeterminate"

## Step 2: the method of joints as a linear system
Take one pin. Bars pull on it along their axes. A bar in tension (force F > 0) pulls the
pin towards the bar's other end. With direction cosines (cx, cy) pointing from this pin to
the other end, the pin feels F * (cx, cy). Writing sum Fx = 0 and sum Fy = 0 for every pin
gives A x = b where x holds all bar forces and support reactions.

In [ ]:
def solve_by_joints(nodes, members, supports, loads):
    """Solve a determinate truss.

    nodes: (j, 2) coordinates in m.  members: list of (i, k) node pairs.
    supports: dict node -> (restrain_x, restrain_y) with 0/1 flags.
    loads: dict node -> (Fx, Fy) in kN (applied to the joint).
    Returns (forces, reactions): forces in kN, tension positive; reactions = list of
    (node, direction 0/1, value in kN).
    """
    nodes = np.asarray(nodes, float)
    j, m = len(nodes), len(members)
    reac = [(n, d) for n, fl in sorted(supports.items()) for d in (0, 1) if fl[d]]
    A = np.zeros((2 * j, m + len(reac)))
    for e, (i, k) in enumerate(members):
        vec = nodes[k] - nodes[i]
        c = vec / np.linalg.norm(vec)
        A[2 * i:2 * i + 2, e] += c      # tension pulls node i towards node k
        A[2 * k:2 * k + 2, e] -= c      # and node k towards node i
    for q, (n, d) in enumerate(reac):
        A[2 * n + d, m + q] = 1.0
    b = np.zeros(2 * j)
    for n, (fx, fy) in loads.items():
        b[2 * n] -= fx                   # loads move to the right-hand side with a minus sign
        b[2 * n + 1] -= fy
    x = np.linalg.solve(A, b)
    return x[:m], [(n, d, float(v)) for (n, d), v in zip(reac, x[m:])]

## Step 3: a three-bar bracket
A bracket on a wall: pin A at the top, B below it, joint C sticks out 2.0 m and carries
the load. Members AC (sloping up to the wall), BC (horizontal) and AB (the vertical wall
bar). A 12 kN load hangs at C. Hand check at C: AC pulls up and in, BC pushes out.

In [ ]:
def bracket(load=12.0, reach=2.0, rise=1.5):
    nodes = [[0.0, rise], [0.0, 0.0], [reach, 0.0]]          # A, B, C
    members = [(0, 2), (1, 2), (0, 1)]                        # AC, BC, AB
    supports = {0: (1, 1), 1: (1, 0)}                         # A pinned, B horizontal roller
    forces, reac = solve_by_joints(nodes, members, supports, {2: (0.0, -load)})
    return forces, reac

## Step 4: a Warren truss
A Warren truss has sloping bars alternately up and down and no verticals. Our example:
span 6 m, bottom joints every 2 m, equilateral triangles of side 2 m (height sqrt(3)
= 1.732 m). That is 7 joints, 11 bars and 3 reactions (a pin and a roller):
m + r = 14 = 2 j. The 10 kN load acts down on the top joint at midspan.

In [ ]:
def warren(side=2.0, panels=3):
    """Nodes and members of a Warren truss with `panels` bottom panels of equilateral triangles."""
    h = side * np.sqrt(3.0) / 2.0
    bottom = [[side * i, 0.0] for i in range(panels + 1)]
    top = [[side * (i + 0.5), h] for i in range(panels)]
    nodes = bottom + top
    nb = panels + 1
    members = [(i, i + 1) for i in range(panels)]                      # bottom chord
    members += [(nb + i, nb + i + 1) for i in range(panels - 1)]       # top chord
    for i in range(panels):                                            # sloping bars
        members += [(i, nb + i), (nb + i, i + 1)]
    return np.array(nodes), members


def warren_example(load=10.0):
    nodes, members = warren()
    supports = {0: (1, 1), 3: (0, 1)}               # pin at the left foot, roller at the right foot
    mid_top = 5                                      # top joint at x = 3 m
    forces, reac = solve_by_joints(nodes, members, supports, {mid_top: (0.0, -load)})
    return nodes, members, forces, reac

## Step 5: method of sections (a second route to the same answer)
Instead of going pin by pin, cut the truss in two and write equilibrium of one piece.
Cut through three bars. Taking moments about the point where two of them meet removes
those two forces, leaving the third.
Left piece, cut at x = 2.5 m: only the left reaction (5 kN up at x = 0) acts on it.
  bottom chord: moment about the top joint at x = 3 m:  F_bottom * h = R * 3
  top chord:    moment about the bottom joint at x = 2 m:  F_top * h = - R * 2

In [ ]:
def sections_midspan(load=10.0, side=2.0):
    """Bottom-chord and top-chord forces (kN) in the middle panel by the method of sections."""
    h = side * np.sqrt(3.0) / 2.0
    r_left = load / 2.0                              # symmetric load: each support carries half
    f_bottom = r_left * 3.0 / h                      # tension
    f_top = -r_left * 2.0 / h                        # compression
    return float(f_bottom), float(f_top)

## Step 6: size the bars for a stress limit
Stress sigma = F / A. If the stress limit is sigma_allow (MPa = N/mm^2), a bar needs an
area A = |F| / sigma_allow. With F in kN: A (mm^2) = 1000 |F| / sigma_allow.
Compression bars also have a *buckling* concern: a long slim bar can bend sideways well
below the stress limit, so in practice they are made thicker. Session 3 checks one case.

In [ ]:
def required_area(force_kn, sigma_mpa):
    """Smallest bar area in mm^2 for a force in kN and an allowable stress in MPa."""
    return 1000.0 * abs(force_kn) / sigma_mpa

## Step 7: see it

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    f, reac = bracket()
    print("bracket forces AC, BC, AB (kN):", np.round(f, 2), " reactions:", [(n, d, round(v, 2)) for n, d, v in reac])
    nodes, members, forces, reac = warren_example()
    print("Warren reactions:", [(n, d, round(v, 2)) for n, d, v in reac])
    for k, ((i, j), fk) in enumerate(zip(members, forces)):
        print(f"  bar {k:2d} ({i}-{j}): {fk:+.2f} kN")
    print("sections (bottom, top) =", np.round(sections_midspan(), 2))
    print("determinacy:", determinacy(len(members), 3, len(nodes)))
    print("largest force", round(float(np.abs(forces).max()), 2), "kN needs",
          round(required_area(np.abs(forces).max(), 150.0), 1), "mm^2 at 150 MPa")

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
    big = np.abs(forces).max()
    for (i, j), fk in zip(members, forces):
        col = "tab:blue" if fk > 1e-9 else ("tab:red" if fk < -1e-9 else "0.6")
        ax1.plot(nodes[[i, j], 0], nodes[[i, j], 1], color=col, lw=1 + 6 * abs(fk) / big)
    ax1.plot(nodes[:, 0], nodes[:, 1], "ko", ms=5)
    ax1.annotate("", xy=(3, 1.732), xytext=(3, 3.2), arrowprops=dict(arrowstyle="->", lw=2))
    ax1.text(3.15, 2.7, "10 kN")
    ax1.set_aspect("equal")
    ax1.set_xlabel("x (m)")
    ax1.set_ylabel("y (m)")
    ax1.set_ylim(-0.6, 3.6)
    ax1.set_title("Warren truss: blue tension, red compression")
    cols = ["tab:blue" if v > 0 else "tab:red" for v in forces]
    ax2.bar(range(len(forces)), forces, color=cols)
    ax2.axhline(0, color="gray", lw=0.8)
    ax2.set_xlabel("bar number")
    ax2.set_ylabel("force (kN, tension positive)")
    ax2.set_title("Force in every bar")
    ax2.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUT / "warren_forces.png", dpi=120)

    fig2, ax3 = plt.subplots(figsize=(6, 4))
    sig = np.linspace(50, 250, 81)
    ax3.plot(sig, [required_area(big, s) for s in sig], color="tab:blue")
    ax3.set_xlabel("allowable stress (MPa)")
    ax3.set_ylabel("area of the most loaded bar (mm^2)")
    ax3.set_title("Higher allowable stress needs less steel (10 kN load)")
    ax3.grid(alpha=0.3)
    fig2.tight_layout()
    fig2.savefig(OUT / "area_vs_stress.png", dpi=120)
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint.

## Coding tasks
1. **Easy.** Print the bracket forces for a load of 18 kN with `bracket(18.0)` and check by hand that the tie carries 30 kN and the strut -24 kN.
2. **Stretch.** Add a function that checks equilibrium of every joint of `warren_example()` and prints the largest residual force.
3. **Explain.** Explain why the vertical bar AB of the bracket carries no force, using the equilibrium of pin B.

## Assessment
- **A. Conceptual.** A truss has 12 joints, 21 bars and 3 support reactions. Is it determinate? What would you expect if one bar were removed, and what if one were added?
- **B. Numerical.** Double the Warren load to 20 kN. Find the sloping bar forces and the middle bottom chord, and the area the chord needs at 150 MPa. Then round the areas up to 120 mm^2 for the chord and 80 mm^2 for the sloping bars: what are the stresses?
- **C. Graph and simulation interpretation.** In the bar chart of the Warren truss, how many bars are in tension, how many in compression, and which group is the top chord in? Which bar has the largest force?
- **D. Design and optimisation challenge.** Design: for the 20 kN load use bars of 120 mm^2 for the chord and 80 mm^2 for the other 10 bars, against 120 mm^2 for all 11 bars. Compare the steel volumes (all bars 2 m long) and say what the buckling concern changes.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    # three-bar bracket, hand calculation at joint C: sin(a) = 1.5/2.5 = 0.6, cos(a) = 0.8
    f, reac = L.bracket(12.0, 2.0, 1.5)
    checks.append(("bracket hand calculation: tie AC = 12/0.6 = +20 kN, strut BC = -20*0.8 = -16 kN",
                   abs(f[0] - 20.0) < 1e-9 and abs(f[1] + 16.0) < 1e-9))
    checks.append(("bracket: the vertical wall bar AB carries zero force", abs(f[2]) < 1e-9))
    rx = {(n, d): v for n, d, v in reac}
    checks.append(("bracket reactions: A gives 12 kN up, A and B balance 16 kN horizontally",
                   abs(rx[(0, 1)] - 12.0) < 1e-9 and abs(rx[(0, 0)] + 16.0) < 1e-9 and abs(rx[(1, 0)] - 16.0) < 1e-9))

    nodes, members, forces, reac = L.warren_example(10.0)
    checks.append(("Warren truss has 7 joints, 11 bars, 3 reactions and m + r = 2 j",
                   len(nodes) == 7 and len(members) == 11 and L.determinacy(11, 3, 7) == "determinate"))
    checks.append(("worked example: the sloping bars carry +-5.77 kN (5 / sin 60)",
                   all(abs(abs(forces[k]) - 5.77) < 0.005 for k in range(5, 11))
                   and abs(5.0 / np.sin(np.radians(60)) - 5.7735) < 1e-3))
    checks.append(("worked example: bottom chord 2.89, 8.66, 2.89 kN (tension) and top chord -5.77, -5.77 kN",
                   np.allclose(forces[:3], [2.887, 8.660, 2.887], atol=0.005) and np.allclose(forces[3:5], -5.774, atol=0.005)))
    rr = {(n, d): v for n, d, v in reac}
    checks.append(("support reactions are 5 kN up at each foot and 0 horizontally",
                   abs(rr[(0, 1)] - 5.0) < 1e-9 and abs(rr[(3, 1)] - 5.0) < 1e-9 and abs(rr[(0, 0)]) < 1e-9))
    fb, ft = L.sections_midspan(10.0)
    checks.append(("method of sections gives the same middle bottom chord (+8.66 kN) and top chord (-5.77 kN)",
                   abs(fb - forces[1]) < 1e-9 and abs(ft - forces[3]) < 1e-9))
    # equilibrium of every joint: sum of member pulls + reaction + load = 0
    res = np.zeros((len(nodes), 2))
    for (i, k), fk in zip(members, forces):
        c = (nodes[k] - nodes[i]) / np.linalg.norm(nodes[k] - nodes[i])
        res[i] += fk * c
        res[k] -= fk * c
    for n, d, v in reac:
        res[n, d] += v
    res[5, 1] -= 10.0
    checks.append(("every joint is in equilibrium (sum Fx = 0 and sum Fy = 0)", np.allclose(res, 0.0, atol=1e-9)))
    # moment equilibrium of the whole truss about the left foot: 10 kN at x = 3 m, reaction 5 kN at x = 6 m
    checks.append(("moments about the left foot balance: 10 * 3 = R_right * 6", abs(10.0 * 3.0 - rr[(3, 1)] * 6.0) < 1e-9))
    checks.append(("linearity: doubling the load doubles every bar force",
                   np.allclose(L.warren_example(20.0)[2], 2 * forces, atol=1e-9)))
    # determinacy classes
    checks.append(("determinacy: one bar removed is a mechanism, one bar added is indeterminate",
                   L.determinacy(10, 3, 7) == "mechanism" and L.determinacy(12, 3, 7) == "indeterminate"))
    checks.append(("a triangle with a pin and a roller is determinate (3 + 3 = 2 * 3)", L.determinacy(3, 3, 3) == "determinate"))
    # design
    checks.append(("worked example: the 8.66 kN bar needs 57.7 mm^2 at 150 MPa",
                   abs(L.required_area(forces[1], 150.0) - 57.735) < 0.01))
    checks.append(("required area is the same for tension and compression of equal size",
                   abs(L.required_area(-8.0, 100.0) - L.required_area(8.0, 100.0)) < 1e-12 and abs(L.required_area(8.0, 100.0) - 80.0) < 1e-9))
    checks.append(("edge: a zero load gives zero force everywhere", np.allclose(L.warren_example(0.0)[2], 0.0)))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
